# Bronze Layer Ingestion — IBM HR Analytics Dataset

## Dataset
**IBM HR Analytics Employee Attrition & Performance** (Kaggle)
- 1,470 employees × 35 features
- Target: `Attrition` (Yes/No)
- Real correlations: OverTime, MonthlyIncome, JobLevel, StockOptionLevel, etc.

## Instructions
1. Download CSV from: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset
2. Upload `WA_Fn-UseC_-HR-Employee-Attrition.csv` to Databricks:
   - Navigate to **Catalog → workspace → bronze → Volumes → source_systems**
   - Or upload via: **File → Upload data to DBFS**
   - Place at: `/Volumes/workspace/bronze/source_systems/ibm_hr_attrition.csv`
3. Run the cells below to ingest into Delta table

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# INGESTION CONFIG — IBM HR Analytics Dataset
# ═══════════════════════════════════════════════════════════════════════════════
# Single CSV file → single bronze Delta table
# This replaces the previous 4-table ingestion from the HRDataset_v14
# ═══════════════════════════════════════════════════════════════════════════════

INGESTION_CONFIG = [
    {
        "path": "/Volumes/workspace/bronze/source_systems/ibm_hr_attrition.csv",
        "table": "ibm_hr_attrition"
    }
]

print("📋 Ingestion config ready:")
for item in INGESTION_CONFIG:
    print(f"   {item['path']} → workspace.bronze.{item['table']}")

# Ingest IBM HR Analytics CSV into Bronze Layer

In [0]:
import re

for item in INGESTION_CONFIG:
    table_name = item["table"]
    file_path = item["path"]
    
    print(f"📥 Ingesting: {file_path}")
    print(f"   → workspace.bronze.{table_name}")
    
    # Read CSV with schema inference
    df = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(file_path)
    )
    
    # Clean column names: replace invalid characters with underscores
    df = df.toDF(*[re.sub(r"[ ,;{}()\n\t=\-]", "_", col) for col in df.columns])
    
    # Write as Delta table
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .saveAsTable(f"workspace.bronze.{table_name}")
    )
    
    row_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM workspace.bronze.{table_name}").collect()[0]["cnt"]
    col_count = len(spark.table(f"workspace.bronze.{table_name}").columns)
    
    print(f"   ✅ {row_count:,} rows × {col_count} columns loaded")

# Show the schema
print("\n📋 Bronze table schema:")
spark.table("workspace.bronze.ibm_hr_attrition").printSchema()

# Quick preview
print("\n📊 Sample data:")
display(spark.table("workspace.bronze.ibm_hr_attrition").limit(5))

Ingesting workspace.bronze.employee_data
Ingesting workspace.bronze.employee_engagement_survey_data
Ingesting workspace.bronze.recruitment_data
Ingesting workspace.bronze.training_and_development_data
